# 面试问题：Mamba 的 Selective SSM 怎样逐步扫描，并如何从零实现可训练版本？

## 可直接复述的回答主线

1. Mamba 的核心不是普通 RNN，而是输入相关的离散化步长 delta、写入 B 和读出 C，让模型逐 token 决定记什么、忘什么。
2. 连续状态矩阵 A 必须稳定参数化；常见做法是 A=-exp(A_log)，离散衰减 exp(delta*A) 因而落在零到一。
3. 因果 depthwise convolution 先提取局部模式，selective scan 再承载长程状态，门控分支控制最终输出。
4. 扫描递推可写成 h_t=exp(delta_t A)h_{t-1}+delta_t B_t u_t，y_t=C_t h_t+D u_t。
5. 本例不用现成 Mamba 包，显式写出每个时间步的状态更新，并对每一时刻的最后有效传感值做监督。
6. 评测同时展示同数据 baseline MSE、逐样本时间序列、delta/decay/state norm、梯度和正 A 爆炸反例。
7. 生产实现还需 fused parallel scan、长序列数值精度、chunk state、padding/reset 语义、吞吐和漂移监控。

后续实验会在同一批可读输入上依次展示基线、手写核心机制、训练过程、逐样本结果、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是 8 条长度为 6 的仓库传感器流。每个事件包含 measurement 和 accepted 标志；被拒绝的尖峰不应覆盖记忆，目标是在每个时刻输出最近一次 accepted 的真实读数。实验逐条打印事件流，并用“直接相信当前读数”作同口径 MSE 基线。

In [1]:
import math  # 汇总梯度、状态和误差指标。
import warnings  # 过滤本地 PyTorch 的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦状态空间过程。
import torch  # 使用基础张量和自动微分手写 selective scan。
torch.manual_seed(441)  # 固定参数初始化和训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程以提高复现性。
sensor_sequences = [[(0.1, 1), (0.2, 1), (5.0, 0), (0.4, 1), (7.0, 0), (8.0, 0)], [(1.0, 1), (9.0, 0), (1.2, 1), (8.0, 0), (1.4, 1), (7.0, 0)], [(-0.5, 1), (4.0, 0), (-0.2, 1), (3.0, 0), (2.0, 0), (1.0, 0)], [(0.8, 1), (0.9, 1), (6.0, 0), (5.0, 0), (1.1, 1), (9.0, 0)], [(-1.0, 1), (-0.8, 1), (7.5, 0), (-0.6, 1), (6.5, 0), (5.5, 0)], [(0.3, 1), (8.0, 0), (7.0, 0), (0.6, 1), (0.7, 1), (9.0, 0)], [(1.5, 1), (6.0, 0), (1.3, 1), (5.0, 0), (1.0, 1), (4.0, 0)], [(-0.2, 1), (3.5, 0), (0.0, 1), (4.5, 0), (0.2, 1), (5.5, 0)]]  # 定义八条含有效读数和异常尖峰的真实事件流。
def running_last_accepted(sequence):  # 为一条事件流生成逐时刻监督标签。
    outputs = []  # 保存到当前为止最近的有效读数。
    remembered = 0.0  # 初始化状态占位值。
    for measurement, accepted in sequence:  # 按时间顺序处理传感事件。
        if accepted == 1:  # 判断当前读数是否通过质量门禁。
            remembered = measurement  # 仅用有效读数覆盖记忆。
        outputs.append(remembered)  # 把当前正确状态作为该时刻标签。
    return outputs  # 返回完整时序目标。
target_sequences = [running_last_accepted(sequence) for sequence in sensor_sequences]  # 计算八条真实监督序列。
raw_measurements = torch.tensor([[measurement for measurement, accepted in sequence] for sequence in sensor_sequences], dtype=torch.float32)  # 构造原始测量批张量。
accepted_flags = torch.tensor([[accepted for measurement, accepted in sequence] for sequence in sensor_sequences], dtype=torch.float32)  # 构造质量门禁批张量。
features = torch.stack([raw_measurements / 10.0, accepted_flags], dim=-1)  # 把缩放测量值和 accepted 组成两维事件特征。
targets = torch.tensor(target_sequences, dtype=torch.float32)  # 构造逐时间步最后有效值标签。
print(f"输入张量shape={tuple(features.shape)}，字段=[measurement/10, accepted]")  # 展示批量、长度与字段 schema。
for index, (sequence, target) in enumerate(zip(sensor_sequences, target_sequences)):  # 逐条展示至少六个可读样例。
    event_text = " -> ".join(f"{measurement:+.1f}({'收' if accepted else '拒'})" for measurement, accepted in sequence)  # 把事件流转成人类可读文本。
    print(f"样本{index + 1}: {event_text} | last-accepted target={target}")  # 输出真实输入与逐步监督。

输入张量shape=(8, 6, 2)，字段=[measurement/10, accepted]
样本1: +0.1(收) -> +0.2(收) -> +5.0(拒) -> +0.4(收) -> +7.0(拒) -> +8.0(拒) | last-accepted target=[0.1, 0.2, 0.2, 0.4, 0.4, 0.4]
样本2: +1.0(收) -> +9.0(拒) -> +1.2(收) -> +8.0(拒) -> +1.4(收) -> +7.0(拒) | last-accepted target=[1.0, 1.0, 1.2, 1.2, 1.4, 1.4]
样本3: -0.5(收) -> +4.0(拒) -> -0.2(收) -> +3.0(拒) -> +2.0(拒) -> +1.0(拒) | last-accepted target=[-0.5, -0.5, -0.2, -0.2, -0.2, -0.2]
样本4: +0.8(收) -> +0.9(收) -> +6.0(拒) -> +5.0(拒) -> +1.1(收) -> +9.0(拒) | last-accepted target=[0.8, 0.9, 0.9, 0.9, 1.1, 1.1]
样本5: -1.0(收) -> -0.8(收) -> +7.5(拒) -> -0.6(收) -> +6.5(拒) -> +5.5(拒) | last-accepted target=[-1.0, -0.8, -0.8, -0.6, -0.6, -0.6]
样本6: +0.3(收) -> +8.0(拒) -> +7.0(拒) -> +0.6(收) -> +0.7(收) -> +9.0(拒) | last-accepted target=[0.3, 0.3, 0.3, 0.6, 0.7, 0.7]
样本7: +1.5(收) -> +6.0(拒) -> +1.3(收) -> +5.0(拒) -> +1.0(收) -> +4.0(拒) | last-accepted target=[1.5, 1.5, 1.3, 1.3, 1.0, 1.0]
样本8: -0.2(收) -> +3.5(拒) -> +0.0(收) -> +4.5(拒) -> +0.2(收) -> +5.5(拒) | last-accepted 

## 2. Baseline / 基线：直接使用当前 measurement

这个朴素策略忽略 accepted 标志，因此异常尖峰会立即污染输出。它和后续 Mamba 在相同 8×6 个时间点上计算 MSE，并额外报告每条流最后时刻的 MSE。

In [2]:
baseline_predictions = raw_measurements.clone()  # 让 baseline 无条件相信每个当前传感读数。
baseline_squared_errors = (baseline_predictions - targets).pow(2)  # 计算每个时间点的平方误差。
baseline_mse = baseline_squared_errors.mean().item()  # 汇总全序列同口径 MSE。
baseline_final_mse = baseline_squared_errors[:, -1].mean().item()  # 汇总最后时刻记忆误差。
print(f"Baseline all-step MSE={baseline_mse:.4f} final-step MSE={baseline_final_mse:.4f}")  # 输出朴素当前值基线。
for index in range(len(sensor_sequences)):  # 逐流展示 baseline 的错误落点。
    print(f"样本{index + 1} baseline={baseline_predictions[index].tolist()} target={targets[index].tolist()} mse={baseline_squared_errors[index].mean().item():.4f}")  # 输出预测、目标和逐样本 MSE。

Baseline all-step MSE=17.5523 final-step MSE=37.0200
样本1 baseline=[0.10000000149011612, 0.20000000298023224, 5.0, 0.4000000059604645, 7.0, 8.0] target=[0.10000000149011612, 0.20000000298023224, 0.20000000298023224, 0.4000000059604645, 0.4000000059604645, 0.4000000059604645] mse=20.7267
样本2 baseline=[1.0, 9.0, 1.2000000476837158, 8.0, 1.399999976158142, 7.0] target=[1.0, 1.0, 1.2000000476837158, 1.2000000476837158, 1.399999976158142, 1.399999976158142] mse=23.6000
样本3 baseline=[-0.5, 4.0, -0.20000000298023224, 3.0, 2.0, 1.0] target=[-0.5, -0.5, -0.20000000298023224, -0.20000000298023224, -0.20000000298023224, -0.20000000298023224] mse=6.1283
样本4 baseline=[0.800000011920929, 0.8999999761581421, 6.0, 5.0, 1.100000023841858, 9.0] target=[0.800000011920929, 0.8999999761581421, 0.8999999761581421, 0.8999999761581421, 1.100000023841858, 1.100000023841858] mse=17.5383
样本5 baseline=[-1.0, -0.800000011920929, 7.5, -0.6000000238418579, 6.5, 5.5] target=[-1.0, -0.800000011920929, -0.80000001192092

## 3. 底层实现：因果 Depthwise Conv 与显式 Selective Scan

`SelectiveScan` 不调用任何现成 SSM。它真的按时间循环，保存每一步的 `delta`、离散衰减、hidden state 和读出；`A=-exp(A_log)` 保证每个通道的连续动力系统稳定。

In [3]:
class CausalDepthwiseConv1d(torch.nn.Module):  # 手写按通道独立的一维因果卷积。
    def __init__(self, channels, kernel_size=3):  # 初始化每个通道的局部时间核。
        super().__init__()  # 注册可学习参数。
        self.kernel_size = kernel_size  # 保存卷积窗口长度。
        self.weight = torch.nn.Parameter(torch.randn(channels, kernel_size) * 0.08)  # 为每个 hidden 通道创建独立卷积核。
        self.bias = torch.nn.Parameter(torch.zeros(channels))  # 创建每通道偏置。
    def forward(self, values):  # 对批乘时间乘通道输入执行左侧 padding 卷积。
        channel_first = values.transpose(1, 2)  # 转成批乘通道乘时间便于切窗。
        padded = torch.nn.functional.pad(channel_first, (self.kernel_size - 1, 0))  # 只在过去方向补零以禁止未来泄漏。
        outputs = []  # 保存每个时间点的卷积结果。
        for time_index in range(values.shape[1]):  # 从左到右读取因果窗口。
            window = padded[:, :, time_index:time_index + self.kernel_size]  # 取得当前及过去的固定长度片段。
            convolved = (window * self.weight[None, :, :]).sum(dim=-1) + self.bias  # 按通道执行乘加而不混合通道。
            outputs.append(convolved)  # 保存当前时间步局部特征。
        return torch.stack(outputs, dim=1)  # 恢复批乘时间乘通道布局。
class SelectiveScan(torch.nn.Module):  # 显式实现输入依赖的状态空间递推。
    def forward(self, inputs, delta, write, read, matrix_a, direct):  # 接收每步动态参数并扫描序列。
        batch, length, channels = inputs.shape  # 读取批量、时间和 hidden 通道数。
        state_size = matrix_a.shape[1]  # 读取每通道内部状态维度。
        state = torch.zeros(batch, channels, state_size, dtype=inputs.dtype)  # 初始化所有样本的 SSM hidden state。
        outputs = []  # 保存每个时间步的读出。
        state_trace = []  # 保存完整状态轨迹供解释。
        decay_trace = []  # 保存离散衰减轨迹供稳定性检查。
        for time_index in range(length):  # 严格从左到右执行因果扫描。
            step_delta = delta[:, time_index]  # 读取当前每通道离散步长。
            decay = torch.exp(step_delta[:, :, None] * matrix_a[None, :, :])  # 将稳定连续 A 离散化为零到一衰减。
            input_update = step_delta[:, :, None] * write[:, time_index, None, :] * inputs[:, time_index, :, None]  # 计算输入相关的状态写入量。
            state = decay * state + input_update  # 同时遗忘旧状态并写入当前事件。
            output = (state * read[:, time_index, None, :]).sum(dim=-1) + direct[None, :] * inputs[:, time_index]  # 用动态 C 和直接项读取状态。
            outputs.append(output)  # 保存当前 SSM 输出。
            state_trace.append(state)  # 保存当前完整 hidden state。
            decay_trace.append(decay)  # 保存当前离散稳定系数。
        return torch.stack(outputs, dim=1), torch.stack(state_trace, dim=1), torch.stack(decay_trace, dim=1)  # 返回输出、状态和衰减序列。
class MambaMixer(torch.nn.Module):  # 组合局部卷积、动态参数、scan 和门控输出。
    def __init__(self, dimension=16, state_size=4):  # 初始化一个简化 Mamba block。
        super().__init__()  # 注册所有可学习部件。
        self.dimension = dimension  # 保存 hidden 通道数。
        self.state_size = state_size  # 保存每通道内部状态数。
        self.input_projection = torch.nn.Linear(dimension, 2 * dimension)  # 同时产生内容分支和 gate 分支。
        self.causal_convolution = CausalDepthwiseConv1d(dimension, kernel_size=3)  # 创建短程因果 depthwise conv。
        self.parameter_projection = torch.nn.Linear(dimension, dimension + 2 * state_size)  # 从当前输入产生 delta、B 和 C。
        self.a_log = torch.nn.Parameter(torch.log(torch.arange(1, state_size + 1, dtype=torch.float32)).repeat(dimension, 1))  # 初始化不同时间尺度的连续状态矩阵。
        self.direct = torch.nn.Parameter(torch.ones(dimension))  # 创建 input-to-output 的直接通路 D。
        self.output_projection = torch.nn.Linear(dimension, dimension)  # 把门控 scan 输出混合回 residual 维度。
        self.scan = SelectiveScan()  # 创建显式时间递推器。
    def forward(self, hidden):  # 执行完整简化 Mamba mixer。
        projected = self.input_projection(hidden)  # 一次投影产生两条分支。
        content, gate = projected.chunk(2, dim=-1)  # 拆出 SSM 内容和输出 gate。
        convolved = torch.nn.functional.silu(self.causal_convolution(content))  # 提取只依赖历史的局部模式。
        dynamic = self.parameter_projection(hidden)  # 从每个实际事件产生动态 scan 参数。
        delta_logits, write, read = torch.split(dynamic, [self.dimension, self.state_size, self.state_size], dim=-1)  # 拆分 delta、B 和 C。
        delta = torch.nn.functional.softplus(delta_logits) + 1.0e-3  # 保证每个离散步长严格为正。
        matrix_a = -torch.exp(self.a_log)  # 用负指数保证连续状态矩阵严格稳定。
        scanned, state_trace, decay_trace = self.scan(convolved, delta, write, read, matrix_a, self.direct)  # 执行逐时间步选择性状态递推。
        gated = scanned * torch.sigmoid(gate)  # 用输入相关 gate 控制哪些状态读出通过。
        output = self.output_projection(gated)  # 混合各 hidden 通道形成 block 输出。
        debug = {"content": content, "convolved": convolved, "delta": delta, "write": write, "read": read, "matrix_a": matrix_a, "state_trace": state_trace, "decay_trace": decay_trace, "gate": torch.sigmoid(gate), "scanned": scanned}  # 保存关键选择性中间量。
        return output, debug  # 返回 mixer 输出与完整扫描证据。
class TinyMambaRegressor(torch.nn.Module):  # 用单个 Mamba block 预测逐时刻最后有效值。
    def __init__(self, input_dimension=2, dimension=16, state_size=4):  # 初始化输入投影、Mamba 和回归头。
        super().__init__()  # 注册完整模型参数。
        self.input_projection = torch.nn.Linear(input_dimension, dimension)  # 把两维事件映射到 hidden。
        self.mixer = MambaMixer(dimension, state_size)  # 创建选择性状态空间核心。
        self.norm = torch.nn.LayerNorm(dimension)  # 稳定 residual 输出尺度。
        self.head = torch.nn.Linear(dimension, 1)  # 把每时刻 hidden 映射成标量读数。
    def forward(self, event_features):  # 对完整事件流进行因果建模。
        hidden = torch.tanh(self.input_projection(event_features))  # 编码 measurement 和 accepted 标志。
        mixed, debug = self.mixer(hidden)  # 运行局部卷积与 selective scan。
        predictions = self.head(self.norm(hidden + mixed)).squeeze(-1)  # 从 residual hidden 逐时刻回归最后有效值。
        debug["input_hidden"] = hidden  # 保存进入 Mamba 前的事件表示。
        debug["mixed_hidden"] = mixed  # 保存 Mamba 输出供对照。
        return predictions, debug  # 返回逐时刻预测和扫描中间量。
model = TinyMambaRegressor()  # 创建待训练的手写 Mamba 回归器。
print(f"TinyMamba parameters={sum(parameter.numel() for parameter in model.parameters())}")  # 展示小型教学模型规模。

TinyMamba parameters=1465


## 4. 真实训练与扫描中间量

对全部 48 个时间点计算 MSE 并真实反向传播。训练后打印 accepted 与 rejected 事件上的平均 delta、每步 decay 范围、状态范数和 gate；这些数值展示模型实际在改变状态，而不是只通过断言声称实现了扫描。

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.008)  # 创建覆盖卷积、动态参数、SSM 和回归头的优化器。
training_history = []  # 保存真实 loss、梯度和 MSE 学习曲线。
for step in range(1201):  # 在八条受控流上执行真实优化。
    model.train()  # 开启训练模式。
    predictions, debug = model(features)  # 运行完整因果卷积和 selective scan。
    loss = torch.nn.functional.mse_loss(predictions, targets)  # 对全部时间步计算回归误差。
    optimizer.zero_grad(set_to_none=True)  # 清除上一轮参数梯度。
    loss.backward()  # 穿过显式时间递推执行反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总当前梯度二范数。
    optimizer.step()  # 用 Adam 更新所有底层参数。
    if step % 300 == 0 or step == 1200:  # 定期保存可读学习轨迹。
        training_history.append({"step": step, "loss": float(loss.item()), "gradient_norm": gradient_norm})  # 记录当前优化证据。
model.eval()  # 切换到确定性评估模式。
with torch.no_grad():  # 计算更新后的最终模型结果。
    mamba_predictions, final_debug = model(features)  # 获得逐时刻预测和扫描状态。
mamba_squared_errors = (mamba_predictions - targets).pow(2)  # 计算每个时间点平方误差。
mamba_mse = mamba_squared_errors.mean().item()  # 汇总和 baseline 相同的全序列 MSE。
mamba_final_mse = mamba_squared_errors[:, -1].mean().item()  # 汇总最后时刻记忆 MSE。
accepted_delta = final_debug["delta"][accepted_flags.bool()].mean().item()  # 统计有效读数上的平均输入相关步长。
rejected_delta = final_debug["delta"][~accepted_flags.bool()].mean().item()  # 统计异常读数上的平均输入相关步长。
state_norm_trace = final_debug["state_trace"][0].flatten(1).norm(dim=1)  # 计算样本一每步完整状态范数。
decay_minimum = final_debug["decay_trace"].min().item()  # 读取全部离散衰减的最小值。
decay_maximum = final_debug["decay_trace"].max().item()  # 读取全部离散衰减的最大值。
print("TinyMamba训练轨迹=", training_history)  # 输出 loss 与梯度真实演化。
print(f"delta mean accepted={accepted_delta:.4f} rejected={rejected_delta:.4f}")  # 展示输入对离散步长的实际影响。
print(f"decay range=[{decay_minimum:.6f}, {decay_maximum:.6f}] A range=[{final_debug['matrix_a'].min().item():.4f}, {final_debug['matrix_a'].max().item():.4f}]")  # 展示稳定 A 产生的零到一离散衰减。
print("样本1 state norm trace=", torch.round(state_norm_trace * 1000) / 1000)  # 展示状态随事件逐步变化。
print("样本1 mean gate trace=", torch.round(final_debug["gate"][0].mean(dim=1) * 1000) / 1000)  # 展示每时刻输出门控强度。

TinyMamba训练轨迹= [{'step': 0, 'loss': 0.6060988306999207, 'gradient_norm': 1.6821582499076395}, {'step': 300, 'loss': 0.0004920904175378382, 'gradient_norm': 0.0020850207924249242}, {'step': 600, 'loss': 0.00010473944712430239, 'gradient_norm': 0.059070257875303345}, {'step': 900, 'loss': 1.8513630493544042e-05, 'gradient_norm': 0.00022951116950680075}, {'step': 1200, 'loss': 8.445643288723659e-06, 'gradient_norm': 0.00011041309898727306}]
delta mean accepted=0.9914 rejected=0.3883
decay range=[0.000000, 0.963432] A range=[-16.3331, -0.5247]
样本1 state norm trace= tensor([1.1660, 0.8670, 0.8160, 1.3080, 1.1150, 1.2830])
样本1 mean gate trace= tensor([0.6380, 0.6390, 0.6390, 0.6400, 0.6430, 0.6450])


## 5. 逐样本结果与结果解读

下面不是只报一个平均值：每条事件流都列出 baseline、Mamba 和正确状态轨迹，并在最后时刻展示是否真正抵抗了 rejected spike。

In [5]:
print("sample  baseline_mse  mamba_mse  last_raw  last_target  last_prediction")  # 输出逐样本误差表头。
for index in range(len(sensor_sequences)):  # 逐流检查模型输出。
    print(f"{index + 1:<6} {baseline_squared_errors[index].mean().item():>12.4f} {mamba_squared_errors[index].mean().item():>10.4f} {raw_measurements[index, -1].item():>9.3f} {targets[index, -1].item():>12.3f} {mamba_predictions[index, -1].item():>15.3f}")  # 输出同口径误差和最终记忆值。
    print("       baseline=", [round(value, 3) for value in baseline_predictions[index].tolist()])  # 展示当前值 baseline 的完整轨迹。
    print("       mamba   =", [round(value, 3) for value in mamba_predictions[index].tolist()])  # 展示手写模型完整预测轨迹。
    print("       target  =", [round(value, 3) for value in targets[index].tolist()])  # 展示正确最后有效值轨迹。
print(f"结果解读：all-step MSE baseline={baseline_mse:.4f} -> Mamba={mamba_mse:.4f}；final-step MSE {baseline_final_mse:.4f} -> {mamba_final_mse:.4f}。")  # 汇总同一数据同一指标收益。

sample  baseline_mse  mamba_mse  last_raw  last_target  last_prediction
1           20.7267     0.0000     8.000        0.400           0.400
       baseline= [0.1, 0.2, 5.0, 0.4, 7.0, 8.0]
       mamba   = [0.101, 0.21, 0.198, 0.395, 0.405, 0.4]
       target  = [0.1, 0.2, 0.2, 0.4, 0.4, 0.4]
2           23.6000     0.0000     7.000        1.400           1.398
       baseline= [1.0, 9.0, 1.2, 8.0, 1.4, 7.0]
       mamba   = [1.0, 0.997, 1.201, 1.202, 1.4, 1.398]
       target  = [1.0, 1.0, 1.2, 1.2, 1.4, 1.4]
3            6.1283     0.0000     1.000       -0.200          -0.200
       baseline= [-0.5, 4.0, -0.2, 3.0, 2.0, 1.0]
       mamba   = [-0.507, -0.499, -0.201, -0.197, -0.2, -0.2]
       target  = [-0.5, -0.5, -0.2, -0.2, -0.2, -0.2]
4           17.5383     0.0000     9.000        1.100           1.100
       baseline= [0.8, 0.9, 6.0, 5.0, 1.1, 9.0]
       mamba   = [0.801, 0.894, 0.902, 0.9, 1.1, 1.1]
       target  = [0.8, 0.9, 0.9, 0.9, 1.1, 1.1]
5           26.0850     0.0

## 6. 失败案例与修正：A 为正导致状态指数爆炸

在相同常量输入、相同 delta 和相同初始状态下，把 A 错写成 `+0.4` 会令离散系数 `exp(delta*A)>1`；稳定参数化 `A=-exp(A_log)` 则让旧状态衰减。下面真的递推 12 步，而不是只给公式。

In [6]:
def scalar_state_trace(matrix_a, steps=12, delta=1.0, update=1.0):  # 运行一维 SSM 状态递推用于稳定性对照。
    state = torch.tensor(0.0)  # 从零状态开始。
    trace = []  # 保存每一步真实状态值。
    for _ in range(steps):  # 连续处理相同输入事件。
        state = torch.exp(torch.tensor(delta * matrix_a)) * state + update  # 应用与 selective scan 相同的离散递推。
        trace.append(float(state.item()))  # 保存当前状态便于观察增长速度。
    return trace  # 返回完整十二步轨迹。
unstable_trace = scalar_state_trace(0.4)  # 故意使用正 A 复现指数增长。
stable_trace = scalar_state_trace(-0.4)  # 使用负 A 让历史贡献逐渐衰减。
print("错误行为：A=+0.4 state trace=", [round(value, 3) for value in unstable_trace])  # 展示状态快速放大。
print("修正行为：A=-0.4 state trace=", [round(value, 3) for value in stable_trace])  # 展示状态收敛到有限值。
print(f"第12步放大比={unstable_trace[-1] / stable_trace[-1]:.2f}x")  # 量化错误符号造成的数值风险。

错误行为：A=+0.4 state trace= [1.0, 2.492, 4.717, 8.037, 12.991, 20.38, 31.403, 47.847, 72.38, 108.978, 163.576, 245.027]
修正行为：A=-0.4 state trace= [1.0, 1.67, 2.12, 2.421, 2.623, 2.758, 2.849, 2.91, 2.95, 2.978, 2.996, 3.008]
第12步放大比=81.45x


## 7. 生产边界

显式 Python 循环便于学习但不能代表生产吞吐。真实 Mamba 需要 associative/fused scan、GPU kernel 与内存布局优化、长序列 chunk state 的正确衔接、padding 边界清零、FP16/BF16 下指数稳定性、梯度裁剪、流式状态版本，以及按 accepted/rejected、序列长度和设备监控误差、状态范数与延迟。

In [7]:
mamba_diagnostics = {"sequences": len(sensor_sequences), "events": int(features.shape[0] * features.shape[1]), "baseline_mse": baseline_mse, "mamba_mse": mamba_mse, "baseline_final_mse": baseline_final_mse, "mamba_final_mse": mamba_final_mse, "initial_loss": training_history[0]["loss"], "final_loss": training_history[-1]["loss"], "decay_range": (decay_minimum, decay_maximum), "unstable_final_state": unstable_trace[-1], "stable_final_state": stable_trace[-1]}  # 汇总数据、优化、选择性状态和失败指标。
print("生产监控快照：", mamba_diagnostics)  # 输出流式状态空间系统应持续观察的信号。

生产监控快照： {'sequences': 8, 'events': 48, 'baseline_mse': 17.552291870117188, 'mamba_mse': 8.419134246651083e-06, 'baseline_final_mse': 37.02000045776367, 'mamba_final_mse': 3.7477193473023362e-06, 'initial_loss': 0.6060988306999207, 'final_loss': 8.445643288723659e-06, 'decay_range': (1.1678670725802365e-13, 0.9634323120117188), 'unstable_final_state': 245.02723693847656, 'stable_final_state': 3.008281946182251}


## 8. 最小回归测试

最后一格只保护真实事件规模、反向传播、同指标收益、稳定参数化、状态轨迹和失败复现。

In [8]:
assert len(sensor_sequences) >= 5 and features.shape == (8, 6, 2) and targets.shape == (8, 6)  # 保证案例包含足够多的真实时序事件。
assert training_history[-1]["loss"] < training_history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in training_history)  # 保证显式 scan 参与真实 backward 学习。
assert mamba_mse < baseline_mse and mamba_final_mse < baseline_final_mse  # 保证相同数据相同 MSE 下优于当前值基线。
assert mamba_mse < 0.10 and mamba_final_mse < 0.10  # 保证模型实际学到最近有效值而非仅略微改善。
assert bool((final_debug["matrix_a"] < 0.0).all()) and 0.0 < decay_minimum <= decay_maximum <= 1.0  # 保证稳定 A 和离散衰减范围正确。
assert unstable_trace[-1] > stable_trace[-1] * 10.0 and state_norm_trace.numel() == features.shape[1]  # 保证正 A 爆炸与逐步状态证据可复现。